## One-stage anchor-based detector: RetinaNet
HW2, notebook 3/5 (Kaggle GPU). Writes `results_03_*.csv`, `final_03.pt` and `submission_03_retinanet.csv` to `/kaggle/working`.

**Needs on Kaggle:** GPU accelerator (T4 / P100), **Internet ON** (pretrained weights are downloaded), inputs: the competition + the output of notebook 01 (or the dataset made from it). No API keys.

**Shared protocol (notebooks 02-05)**
- Data: frames exported by notebook 01 (every 5th frame, ignored regions grayed out). Split **by clip**: 5 GroupKFold folds (grouped by source flight) from notebook 01.
- Cross-validation: every config is trained on `FOLDS_TO_RUN` folds and scored on the held-out clips with the COCO evaluator (ignored regions = `iscrowd`, up to 500 boxes per image). Tables show mean and std over folds.
- Final: the best config (mean mAP) is retrained on all training frames, checked once on the official `val` clips (original resolution, every 5th frame) and used to write the test submission.
- Same metrics everywhere: mAP, AP50, AP_small / medium / large (COCO size buckets), and GPU inference ms per image.

**Model and why**
- torchvision **RetinaNet ResNet50-FPN**, COCO-pretrained, new 8-class head. Same backbone, FPN, input size, optimizer and schedule as Faster R-CNN in notebook 2, so the only difference is the detection head (dense anchors + focal loss instead of RPN + RoI head). It runs at P3-P7 (stride 8-128), which is why the smallest anchors matter.
- No proposal stage: every anchor is classified directly, so the positive/background ratio is extreme. Focal loss down-weights the easy background anchors (that is its role); the anchor/IoU settings below decide how many anchors are positives for tiny objects.

**Configs (each adds one idea to the previous)**
- `C0_default`: torchvision defaults (anchors 32-512 x 3 scales x 3 ratios, positive IoU 0.5, negative 0.4).
- `C1_anchors_iou`: anchor sizes from the GT box-size quantiles (3 scales per level, top scale reaches the 99th percentile) and lower assignment thresholds 0.4 / 0.3 (a few pixels of shift already drops IoU a lot for tiny boxes, so with default thresholds many small objects get no positive anchor).
- `C2_giou_loss`: C1 + GIoU instead of L1 for box regression (torchvision's built-in `giou` option of the regression head).

In [1]:
import os, io, json, time, math, random, shutil, contextlib
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

SEED = 0
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> turn on a GPU in the session options")

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")

def find_all(pattern, depth=5):
    # search attached inputs at increasing nesting depth (Kaggle nests inputs differently per source)
    for d in range(depth):
        hit = sorted(INPUT.glob("/".join(["*"] * d + [pattern])))
        if hit:
            return hit
    return []

def find(pattern):
    hit = find_all(pattern)
    if not hit:
        raise FileNotFoundError(f"{pattern} not found under {INPUT}: check the attached inputs")
    return hit[0]

PREP = find("vd_prep/config.json").parent            # output of notebook 01
CFG = json.load(open(PREP / "config.json"))
TEST_CSV = find("test_images.csv"); ROOT = TEST_CSV.parent
VAL_DIR = find("val/annotations_coco.json").parent
FOLD = CFG["folds"]                                   # train clip -> fold id (GroupKFold by scene, from notebook 01)
METRICS = ["mAP", "AP50", "AP_s", "AP_m", "AP_l", "ms_per_img"]

DEVICE = "cuda"
EPOCHS, BS, LR = 3, 4, 0.0025         # one epoch is about 7 min on a T4
FOLDS_TO_RUN = [0, 1]                  # folds used for CV (of 5)
MIN_SIZE, MAX_SIZE = 544, 960
VAL_STEP = 5
LABEL_SHIFT = -1                       # RetinaNet has no background channel: labels 0..7 = category id - 1
NB = "03"

GPU: Tesla T4


### Data

In [2]:
def load_gt(path):
    # COCO ground truth; ignored regions (iscrowd=1) are copied to all 8 classes so they ignore detections of any class
    d = json.load(open(path))
    crowd = [a for a in d["annotations"] if a.get("iscrowd")]
    d["annotations"] = [a for a in d["annotations"] if not a.get("iscrowd")]
    nid = max([a["id"] for a in d["annotations"]], default=0) + 1
    for a in crowd:
        for c in range(1, 9):
            d["annotations"].append({**a, "id": nid, "category_id": c, "iscrowd": 1, "area": a["bbox"][2] * a["bbox"][3]})
            nid += 1
    gt = COCO()
    gt.dataset = d
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
    return gt

def coco_eval(gt, dets, img_ids):
    # standard COCO bbox metrics, up to 500 detections per image (as in the leaderboard)
    if not dets:
        return dict(mAP=0.0, AP50=0.0, AP_s=0.0, AP_m=0.0, AP_l=0.0)
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(gt, gt.loadRes(dets), "bbox")
        E.params.imgIds = list(img_ids)
        E.params.maxDets = [1, 10, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats                                  # s[1], s[3:6] already use maxDets[-1] = 500
    prec = E.eval["precision"][:, :, :, 0, -1]   # [IoU, recall, class] for all sizes at 500 dets (pycocotools' s[0] hardcodes 100)
    prec = prec[prec > -1]
    return dict(mAP=float(prec.mean()) if prec.size else -1.0, AP50=s[1], AP_s=s[3], AP_m=s[4], AP_l=s[5])

def write_submission(dets, path):
    # one row per test image: "class score x y w h ..." (0-based class, pixels, top-left corner, max 500 boxes)
    test = pd.read_csv(TEST_CSV)
    by = {}
    for d in dets:
        by.setdefault(d["image_id"], []).append(d)
    rows = []
    for iid in test.image_id:
        top = sorted(by.get(iid, []), key=lambda d: -d["score"])[:500]
        rows.append(" ".join(f"{d['category_id'] - 1} {d['score']:.4f} " + " ".join(f"{v:.1f}" for v in d["bbox"]) for d in top))
    sub = pd.DataFrame({"image_id": test.image_id, "PredictionString": rows})
    sub.to_csv(path, index=False)
    sample = find_all("sample_submission.csv")      # not shipped with this competition's data: ids come from test_images.csv
    if sample:
        assert list(sub.image_id) == list(pd.read_csv(sample[0]).image_id), "image ids differ from sample_submission.csv"
    print(f"saved {path}: {len(sub)} rows, {sum(map(bool, rows))} with detections, {sum(len(r.split()) // 6 for r in rows)} boxes")

In [3]:
# training frames exported by notebook 01 (ignored regions already grayed out)
TJ = json.load(open(PREP / "train_coco.json"))
IMGS = {i["id"]: i for i in TJ["images"]}
GT_TRAIN = load_gt(PREP / "train_coco.json")

def fold_ids(f):
    # (train ids, held-out ids) for fold f; f=None -> all frames for training
    return ([i for i, m in IMGS.items() if FOLD[m["video"]] != f],
            [i for i, m in IMGS.items() if FOLD[m["video"]] == f])

def train_path(i):
    return PREP / "images/train" / IMGS[i]["file_name"]

# official val: raw frames at original resolution, every VAL_STEP-th frame of each clip (for speed)
VAL_GT = load_gt(VAL_DIR / "annotations_coco.json")
_idx = {(p.parent.name, p.name): p for p in (VAL_DIR / "images").rglob("*.jpg")}
_v = pd.DataFrame(VAL_GT.dataset["images"])
_v = _v[(_v.groupby("video").frame_id.rank(method="first").astype(int) - 1) % VAL_STEP == 0]
VAL_ITEMS = [(int(r.id), _idx[(r.video, Path(r.file_name).name)]) for r in _v.itertuples()]
VAL_IDS = [i for i, _ in VAL_ITEMS]
TEST_ITEMS = [(r.image_id, ROOT / r.file_name) for r in pd.read_csv(TEST_CSV).itertuples()]
print(f"{len(IMGS)} train frames | {len(VAL_ITEMS)} val frames | {len(TEST_ITEMS)} test frames")

4868 train frames | 571 val frames | 3811 test frames


In [4]:
import albumentations as A
from torch.utils.data import Dataset, DataLoader

ANNS = {}
for a in TJ["annotations"]:
    if not a.get("iscrowd"):
        ANNS.setdefault(a["image_id"], []).append(a)

# augmentation: flip + photometric only (boxes follow the flip)
AUG = A.Compose([A.HorizontalFlip(p=0.5), A.RandomBrightnessContrast(p=0.3)],
                bbox_params=A.BboxParams(format="coco", label_fields=["labels"], min_visibility=0.3))

def read_rgb(p):
    return cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)

class DetDS(Dataset):
    def __init__(self, ids):
        self.ids = list(ids)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, k):
        iid = self.ids[k]; m = IMGS[iid]; W, H = m["width"], m["height"]
        boxes, labels = [], []
        for a in ANNS.get(iid, []):
            x, y, w, h = a["bbox"]
            x0, y0, x1, y1 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
            if x1 - x0 >= 1 and y1 - y0 >= 1:
                boxes.append([x0, y0, x1 - x0, y1 - y0]); labels.append(a["category_id"] + LABEL_SHIFT)
        o = AUG(image=read_rgb(train_path(iid)), bboxes=boxes, labels=labels)
        b = torch.tensor(o["bboxes"], dtype=torch.float32).reshape(-1, 4)
        b[:, 2:] += b[:, :2]                                    # xywh -> xyxy
        img = torch.from_numpy(o["image"]).permute(2, 0, 1).float() / 255
        return img, {"boxes": b, "labels": torch.tensor(o["labels"], dtype=torch.int64)}

class ImgDS(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, k):
        iid, p = self.items[k]
        return iid, torch.from_numpy(read_rgb(p)).permute(2, 0, 1)

def collate(b):
    return tuple(zip(*b))

def fit(model, ids, epochs=None):
    # SGD + linear warmup + cosine decay, mixed precision
    epochs = epochs or EPOCHS
    dl = DataLoader(DetDS(ids), batch_size=BS, shuffle=True, num_workers=4, collate_fn=collate, drop_last=True)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=1e-4)
    total = epochs * len(dl); warm = max(1, min(300, total // 10))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * s / total)))
    scaler = torch.amp.GradScaler("cuda")
    model.to(DEVICE).train()
    for ep in range(epochs):
        run = 0.0
        for k, (imgs, tgts) in enumerate(tqdm(dl, desc=f"epoch {ep + 1}/{epochs}", leave=False)):
            imgs = [i.to(DEVICE) for i in imgs]
            tgts = [{n: v.to(DEVICE) for n, v in t.items()} for t in tgts]
            with torch.autocast("cuda", dtype=torch.float16):
                loss = sum(model(imgs, tgts).values())
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 10.0)
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
        print(f"  epoch {ep + 1}: mean loss {run / len(dl):.4f}")
    return model

@torch.no_grad()
def detect(model, items, bs=8):
    # returns COCO-style detections in the pixel coordinates of the input images and GPU ms per image
    model.eval()
    dl = DataLoader(ImgDS(items), batch_size=bs, num_workers=4, collate_fn=collate)
    dets, t_model = [], 0.0
    for ids, imgs in tqdm(dl, desc="inference", leave=False):
        imgs = [i.to(DEVICE).float() / 255 for i in imgs]
        torch.cuda.synchronize(); t0 = time.time()
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(imgs)
        torch.cuda.synchronize(); t_model += time.time() - t0
        for iid, o in zip(ids, out):
            b = o["boxes"].float().cpu().numpy(); b[:, 2:] -= b[:, :2]
            for box, s, l in zip(b.tolist(), o["scores"].float().cpu().tolist(), o["labels"].cpu().tolist()):
                dets.append({"image_id": iid, "category_id": l - LABEL_SHIFT, "bbox": box, "score": s})
    return dets, 1000 * t_model / len(items)

def eval_fold(model, ids):
    dets, ms = detect(model, [(i, train_path(i)) for i in ids])
    return {**coco_eval(GT_TRAIN, dets, ids), "ms_per_img": ms}

def anchor_stats():
    # GT box size (sqrt of area) after the model's own resize: this is what the anchors have to cover
    s = []
    for a in TJ["annotations"]:
        if a.get("iscrowd"):
            continue
        m = IMGS[a["image_id"]]
        sc = min(MIN_SIZE / min(m["width"], m["height"]), MAX_SIZE / max(m["width"], m["height"]))
        s.append(math.sqrt(a["bbox"][2] * a["bbox"][3]) * sc)
    return np.quantile(np.array(s), [.05, .25, .5, .75, .95, .99])

### Model builder

In [5]:
from torchvision.models.detection import retinanet_resnet50_fpn, RetinaNet_ResNet50_FPN_Weights
from torchvision.models.detection.anchor_utils import AnchorGenerator

q = anchor_stats()
print("GT box sqrt(area) quantiles 5/25/50/75/95/99 % after resize:", q.round(1))
# 5 levels (P3-P7) x 3 octave scales: the largest scale of the last level reaches the 99th percentile
base = np.geomspace(q[0], q[-1] / 2 ** (2 / 3), 5)
DATA_ANCHORS = tuple(tuple(max(4, int(round(s * 2 ** (k / 3)))) for k in range(3)) for s in base)
print("data-driven anchor sizes per level:", DATA_ANCHORS)

def build(anchors=None, fg=0.5, bg=0.4, giou=False):
    kw = dict(min_size=MIN_SIZE, max_size=MAX_SIZE, score_thresh=0.001, detections_per_img=500, fg_iou_thresh=fg, bg_iou_thresh=bg)
    if anchors:
        kw["anchor_generator"] = AnchorGenerator(sizes=anchors, aspect_ratios=((0.5, 1.0, 2.0),) * 5)
    m = retinanet_resnet50_fpn(weights=RetinaNet_ResNet50_FPN_Weights.COCO_V1, weights_backbone=None, **kw)
    h = m.head.classification_head                     # keep the pretrained conv tower, replace only the final 8-class layer
    h.cls_logits = torch.nn.Conv2d(256, h.num_anchors * 8, kernel_size=3, padding=1)
    torch.nn.init.normal_(h.cls_logits.weight, std=0.01)
    torch.nn.init.constant_(h.cls_logits.bias, -math.log((1 - 0.01) / 0.01))      # focal-loss prior: 1 % foreground
    h.num_classes = 8
    if giou:
        m.head.regression_head._loss_type = "giou"
    return m

CONFIGS = {
    "C0_default":     dict(),
    "C1_anchors_iou": dict(anchors=DATA_ANCHORS, fg=0.4, bg=0.3),
    "C2_giou_loss":   dict(anchors=DATA_ANCHORS, fg=0.4, bg=0.3, giou=True),
}

GT box sqrt(area) quantiles 5/25/50/75/95/99 % after resize: [10.2 16.3 22.9 33.  63.  97.6]
data-driven anchor sizes per level: ((10, 13, 16), (16, 20, 25), (25, 32, 40), (39, 49, 62), (61, 77, 98))


### Cross-validation of the three configs

In [6]:
# every config is trained on the same folds (cross-validation over clips) and scored on the held-out clips
rows = []
for name, cfg in CONFIGS.items():
    for f in FOLDS_TO_RUN:
        tr_ids, va_ids = fold_ids(f)
        seed_all()
        model = build(**cfg)
        t0 = time.time()
        fit(model, tr_ids)
        m = eval_fold(model, va_ids)
        rows.append({"config": name, "fold": f, **m, "train_min": (time.time() - t0) / 60})
        print(name, "fold", f, {k: round(v, 4) for k, v in m.items()})
        del model; import gc; gc.collect(); torch.cuda.empty_cache()
folds_df = pd.DataFrame(rows)
folds_df.to_csv(WORK / f"results_{NB}_folds.csv", index=False)
display(folds_df.groupby("config")[METRICS].agg(["mean", "std"]).round(4))

  epoch 1: mean loss 0.6541
  epoch 2: mean loss 0.4780
  epoch 3: mean loss 0.4417
: Traceback (most recent call last):
AssertionError    : self._shutdown_workers()can only test a child processException ignored in:
    : can only test a child processif w.is_alive():
C0_default fold 0 {'mAP': 0.1478, 'AP50': np.float64(0.28), 'AP_s': np.float64(0.0279), 'AP_m': np.float64(0.1937), 'AP_l': np.float64(0.4602), 'ms_per_img': 40.5633}
  epoch 1: mean loss 0.6596
  epoch 2: mean loss 0.4872
  epoch 3: mean loss 0.4501
C0_default fold 1 {'mAP': 0.1156, 'AP50': np.float64(0.2269), 'AP_s': np.float64(0.0246), 'AP_m': np.float64(0.1824), 'AP_l': np.float64(0.4149), 'ms_per_img': 39.4301}
  epoch 1: mean loss 1.0831
  epoch 2: mean loss 0.7823
  epoch 3: mean loss 0.7239
C1_anchors_iou fold 0 {'mAP': 0.0412, 'AP50': np.float64(0.1384), 'AP_s': np.float64(0.0364), 'AP_m': np.float64(0.0574), 'AP_l': np.float64(0.0233), 'ms_per_img': 39.6456}
  epoch 1: mean loss 1.0780
  epoch 2: mean loss 0.7926

mAP            AP50            AP_s            AP_m  \
                  mean     std    mean     std    mean     std    mean   
config                                                                   
C0_default      0.1317  0.0227  0.2535  0.0375  0.0262  0.0023  0.1881   
C1_anchors_iou  0.0389  0.0032  0.1279  0.0148  0.0336  0.0040  0.0505   
C2_giou_loss    0.0379  0.0027  0.1275  0.0121  0.0341  0.0043  0.0488   

                          AP_l         ms_per_img          
                   std    mean     std       mean     std  
config                                                     
C0_default      0.0079  0.4376  0.0320    39.9967  0.8013  
C1_anchors_iou  0.0097  0.0297  0.0091    39.5191  0.1789  
C2_giou_loss    0.0080  0.0259  0.0118    39.5923  0.0891

**Observations:**
- The default config is clearly best: CV mAP 0.132 ± 0.023 vs 0.039 (C1, data-driven anchors + IoU 0.4 / 0.3) and 0.038 (C2, + GIoU).
- C1 raises AP_small only slightly (0.026 -> 0.034) but loses medium and large objects (AP_m 0.188 -> 0.051, AP_l 0.438 -> 0.030). Its loss also starts much higher (1.08 vs 0.65 after epoch 1). Plausible cause: the pretrained head expects COCO anchor sizes on each FPN level, and smaller anchors plus looser thresholds give many low-quality positives that 3 epochs cannot re-learn. The largest new anchor is 98 px, so large objects only match poorly.
- GIoU on top of C1 changes nothing (0.038 vs 0.039, within fold std), so the regression loss was not the limit.
- Speed is the same for all configs (~40 ms per image).

### Final model: retrain on all frames, check on official val

In [7]:
means = folds_df.groupby("config")[METRICS].mean()
best = means.mAP.idxmax()
print("best config by CV mAP:", best)
seed_all()
final = build(**CONFIGS[best])
fit(final, fold_ids(None)[0])
torch.save(final.state_dict(), WORK / "final_03.pt")

val_dets, val_ms = detect(final, VAL_ITEMS)
final_metrics = {**coco_eval(VAL_GT, val_dets, VAL_IDS), "ms_per_img": val_ms}
display(pd.Series(final_metrics).round(4).to_frame("official val"))

out = means.reset_index().assign(split="cv")
out = pd.concat([out, pd.DataFrame([{"config": best + " (final)", **final_metrics, "split": "val"}])], ignore_index=True)
out.insert(0, "model", "RetinaNet R50-FPN")
out.to_csv(WORK / "results_03_summary.csv", index=False)

best config by CV mAP: C0_default
  epoch 1: mean loss 0.6513
  epoch 2: mean loss 0.4857
  epoch 3: mean loss 0.4496


,official val
mAP,0.1455
AP50,0.2798
AP_s,0.0350
AP_m,0.1432
AP_l,0.2990
ms_per_img,41.1573


**Observations:**
- Final model (C0) on the official val: mAP 0.146, AP50 0.280, AP_small 0.035, 41 ms per image.
- Without a proposal stage every anchor (~100k per image) is classified directly, so background dominates. Focal loss handles this imbalance (training was stable), but small objects stay the weak spot: at 544-960 px most objects are 10-30 px, smaller than the default 32 px anchors, so they get few positive anchors and AP_small is only 0.03.
- Shrinking the anchors and lowering the IoU thresholds should address this in theory, but within 3 epochs it hurt much more than it helped (cell above). A longer schedule or adding small anchors to the defaults, instead of replacing them, would be the next steps.
- The comparison with Faster R-CNN is in the `comparisons` notebook.

### Test predictions and submission

In [8]:
test_dets, _ = detect(final, TEST_ITEMS)
write_submission(test_dets, WORK / "submission_03_retinanet.csv")

saved /kaggle/working/submission_03_retinanet.csv: 3811 rows, 3811 with detections, 1871085 boxes
